In [2]:
from ag_vision.core.survey import AgSurvey
from databricks.sdk import WorkspaceClient
from ag_vision.constants import paths
from uuid import uuid4

In [23]:
# This set of code is only used for the data upload to DB with the CLI.
# You will need and api key to make this work
w = WorkspaceClient(profile="artemis")
DB_PROJECT_DIR = '/Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/test_project'

In [24]:
audio_file = "/Users/Dan/Desktop/test_2_audio.wav"
text_file = "/Users/Dan/Desktop/test_2_text.txt"

In [25]:
ag_serv = AgSurvey(platform='local',
                   survey_key='test.json',
                   audio_key=audio_file,
                   text_key=text_file,
                   db_client=w)

In [26]:
ag_serv.load_audio_data()

In [27]:
ag_serv.load_text_data()

In [28]:
ag_serv.text

'This is a test 2\n\nVl'

In [29]:
ag_serv.audio

array([ 2.0392886e-03,  3.2742163e-03,  3.0562668e-03, ...,
       -1.6612739e-04, -4.2909378e-05,  7.7676050e-05],
      shape=(101577,), dtype=float32)

In [30]:
ag_serv.initialize_survey(collection_date='1/1/1991')

In [31]:
ag_serv.survey

SurveyDataModel(path='test.json', id='1d1a53fc-f735-4838-a568-784d85e56a12', collection_date='1/1/1991', trial_properties=TrialProperties(id=None, name=None, url=None, details=None), protocol_properties=ProtocolProperties(name=None, id=None, url=None), location_properties=Location(id=None, name=None, plot_id=None, plotbook_id=None, raw_string=None, latitude=None, longitude=None, elevation_m=None, crs=None, geometry=None, admin_level_0=None, admin_level_1=None, admin_level_2=None, admin_level_3=None, site=None, grower=None, farm=None, field=None, location=None), agronomic_properties=AgronomicProperties(planting_date=None, season_code=None, crop_type=None, growth_stage=None, soil_color=None, weed_pressure=None, irrigation_level=None, tillage_type=None, fertilizer_level=None, plant_health=None), answers={}, followups={}, audio_files=None, image_files=None, notes=[Notes(message='', author='')])

In [32]:
sid = str(uuid4())
location_path = paths.location_path(project=DB_PROJECT_DIR,
                                    site='test_site',
                                    trial='test_trial',
                                    season='test_season',
                                    field='test_field',
                                    location='test_location')

survey_path = paths.survey_path(location_path=location_path,
                                task='test_task',
                                protocol='test_protocol',
                                date='1/3/34',
                                survey_id=sid,
                                f_name=f"{sid}.json")
print(survey_path)

/Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/test_project/test_site/test_trial/test_season/test_field/test_location/test_task/survey/test_protocol/2034-01-03/0350dd47-2a5c-4081-bb9b-254733da8387/0350dd47-2a5c-4081-bb9b-254733da8387.json


In [33]:
ag_serv.upload_survey_data_to_databricks(db_path=survey_path)

In [34]:
survey_text_path = paths.survey_path(location_path=location_path,
                                     task='test_task',
                                     protocol='test_protocol',
                                     date='1/3/34',
                                     survey_id=sid,
                                     f_name='test.txt')

In [35]:
ag_serv.upload_survey_text_to_databricks(db_path=survey_text_path)


In [36]:
survey_audio_path = paths.survey_path(location_path=location_path,
                                      task='test_task',
                                      protocol='test_protocol',
                                      date='1/3/34',
                                     survey_id=sid,
                                     f_name='test.wav')

ag_serv.upload_survey_audio_to_databricks(db_path=survey_audio_path)

In [37]:
f_list = [survey_path, survey_path]

In [48]:
import pandas as pd
def generate_survey_table(file_list: list, project_index: int = 5) -> pd.DataFrame:
    """

    """
    file_list = list(file_list)

    survey_df = pd.DataFrame({'file_path': file_list})

    survey_df['project'] = [x.split('/')[project_index] for x in file_list]
    survey_df['site'] = [x.split('/')[project_index+1 ] for x in file_list]
    survey_df['trial'] = [x.split('/')[project_index + 2] for x in file_list]
    survey_df['season'] = [x.split('/')[project_index + 3] for x in file_list]
    survey_df['field'] = [x.split('/')[project_index + 4] for x in file_list]
    survey_df['location'] = [x.split('/')[project_index + 5] for x in file_list]
    survey_df['protocol'] = [x.split('/')[project_index + 8] for x in file_list]
    survey_df['collection_date'] = [x.split('/')[project_index + 9] for x in file_list]
    survey_df['survey_id'] = [x.split('/')[-1].replace('.json', '') for x in file_list]

    return survey_df

In [49]:
generate_survey_table(file_list=f_list)

,file_path,project,site,trial,season,field,location,protocol,collection_date,survey_id
0,/Volumes/use1_prod_artemis_catalog_37181949744...,test_project,test_site,test_trial,test_season,test_field,test_location,test_protocol,2034-01-03,0350dd47-2a5c-4081-bb9b-254733da8387
1,/Volumes/use1_prod_artemis_catalog_37181949744...,test_project,test_site,test_trial,test_season,test_field,test_location,test_protocol,2034-01-03,0350dd47-2a5c-4081-bb9b-254733da8387
